In [ ]:
%load_ext autoreload
%autoreload 2

import os
import re
from typing import Dict

import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import pickle
import pandas as pd
import seaborn as sns
import datetime
from pathlib import Path
import matplotlib.colors as mcolors
from scipy import sparse
import scipy.sparse as sp
from matplotlib.patches import Rectangle



In [ ]:
# Process samples
samples = ['KS_TMA_1_0026870', 'KS_TMA_2_0026882', 'KS_TMA_3_0027198', 'KS_TMA_4_0026764', 'KS_TMA_5_0026776',
           'KS_TMA_6_0027092', 'KS_TMA_7_0027079', 'KS_TMA_8_0027273', 'KS_TMA_9_0026831', 'KS_TMA_10_0026828', 
           'KS_TMA_11_0026930', 'KS_TMA_12_0026888', 'KS_TMA_13_0027077', 'KS_TMA_14_0027019', 'KS_TMA_15_0033811',
          'KS_TMA_16_0033809']

adata = sc.read_h5ad(f'../data/KS_adata_preprocessed.h5ad')
adata.obsm["spatial"] = adata.obs[["local_x", "local_y"]].copy().to_numpy()

KS_lytic_genes = ['KSHV.ORF50', 'KSHV.ORF57', 'KSHV.ORF59', 'KSHV.K9', 'KSHV.ORF65']
KS_latent_genes = ['KSHV.ORF71','KSHV.ORF72','KSHV.ORF73',]
KS_K2_gene = ['KSHV.K2']

# Spatial Clustering Analysis of KSHV⁺ CD34⁺ LECs

In [ ]:
"""
Spatial Clustering Analysis of KSHV⁺ CD34⁺ LECs
================================================

Determines whether infected CD34+ LECs are spatially clustered or randomly distributed (supporting independent infection).

Methods:
1. Nearest-Neighbor Distance (NND) analysis
2. Comparison to Complete Spatial Randomness (CSR)
3. Clark-Evans aggregation index (R)
4. Ripley's K function (optional)

Key interpretation:
- R < 1: Clustered (observed NND < expected) → supports expansion
- R ≈ 1: Random → supports independent infection
- R > 1: Dispersed/regular
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy import stats
from joblib import Parallel, delayed
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

# Column names (adjust if different in your adata)
PATIENT_COL = 'path_id'
CORE_COL = 'path_block_core'
STAGE_COL = 'Stage'
NICHE_COL = 'niche_with_tumor_proximity'
CELLTYPE_COL = 'broad_cell_types'
INFECTION_COL = 'infection_status'
CD34_COL = 'CD34_status'  # 'CD34+' or 'CD34-'

# Xenium resolution
PIXEL_PER_UM = 1 / 0.2125  # pixels per micrometer

# Stage order and colors
STAGE_ORDER = ['control', 'patch', 'plaque', 'nodular']
DISEASE_STAGES = ['patch', 'plaque', 'nodular']

STAGE_COLORS = {
    'control': '#D3D3D3',
    'patch': '#51f512',
    'plaque': '#ffdc5e',
    'nodular': '#ed322f',
}

# Niche groups
NICHE_GROUPS = {
    'Tumor-associated': ['Tumor Core', 'Tumor', 'Tumor Boundary'],
    'Vascular-associated': ['TA VEC Stroma', 'VEC Stroma'],
    'Immune-associated': ['Macrophage Immune Stroma', 'T-cell Immune Stroma', 'Immune'],
    'Skin-associated': ['Stroma', 'Basal Dermis', 'Differentiated Epidermis']
}

# Output directory
OUTPUT_DIR = 'figures/spatial_clustering'

# Matplotlib settings
plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.size': 10,
    'figure.dpi': 150,
})


def ensure_output_dir():
    """Create output directory if it doesn't exist."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════════
# CORE SPATIAL ANALYSIS FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def compute_nearest_neighbor_distances(coords: np.ndarray) -> np.ndarray:
    """
    Compute nearest-neighbor distances for a set of points.
    
    Parameters
    ----------
    coords : np.ndarray
        Nx2 array of coordinates
    
    Returns
    -------
    np.ndarray
        Array of nearest-neighbor distances
    """
    if len(coords) < 2:
        return np.array([np.nan])
    
    tree = cKDTree(coords)
    # Query for 2 nearest neighbors (first is self with distance 0)
    distances, _ = tree.query(coords, k=2)
    # Return the second column (distance to nearest non-self neighbor)
    return distances[:, 1]


def compute_expected_nnd_csr(n_points: int, area: float) -> float:
    """
    Compute expected mean nearest-neighbor distance under Complete Spatial Randomness.
    
    Under CSR (Poisson process), expected NND = 0.5 * sqrt(A/n)
    where A = area and n = number of points
    
    Parameters
    ----------
    n_points : int
        Number of points
    area : float
        Area of the region
    
    Returns
    -------
    float
        Expected mean NND under CSR
    """
    if n_points < 2 or area <= 0:
        return np.nan
    
    density = n_points / area
    expected_nnd = 0.5 / np.sqrt(density)
    return expected_nnd


def compute_clark_evans_r(observed_mean_nnd: float, expected_mean_nnd: float) -> float:
    """
    Compute Clark-Evans aggregation index R.
    
    R = observed_mean_NND / expected_mean_NND
    
    R < 1: Clustered
    R = 1: Random (CSR)
    R > 1: Dispersed/regular
    
    Parameters
    ----------
    observed_mean_nnd : float
        Observed mean nearest-neighbor distance
    expected_mean_nnd : float
        Expected mean NND under CSR
    
    Returns
    -------
    float
        Clark-Evans R index
    """
    if expected_mean_nnd == 0 or np.isnan(expected_mean_nnd):
        return np.nan
    return observed_mean_nnd / expected_mean_nnd


def compute_clark_evans_z(observed_mean_nnd: float, n_points: int, area: float) -> Tuple[float, float]:
    """
    Compute Clark-Evans Z statistic and p-value.
    
    Tests whether the observed pattern differs significantly from CSR.
    
    Parameters
    ----------
    observed_mean_nnd : float
        Observed mean nearest-neighbor distance
    n_points : int
        Number of points
    area : float
        Area of the region
    
    Returns
    -------
    tuple
        (Z statistic, p-value)
    """
    if n_points < 2 or area <= 0:
        return np.nan, np.nan
    
    density = n_points / area
    expected_nnd = 0.5 / np.sqrt(density)
    
    # Standard error of mean NND under CSR
    se = 0.26136 / np.sqrt(n_points * density)
    
    if se == 0:
        return np.nan, np.nan
    
    z = (observed_mean_nnd - expected_nnd) / se
    
    # Two-tailed p-value
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    
    return z, p_value


def estimate_core_area(coords: np.ndarray, method: str = 'convex_hull') -> float:
    """
    Estimate the area of a TMA core from cell coordinates.
    
    Parameters
    ----------
    coords : np.ndarray
        Nx2 array of coordinates
    method : str
        'convex_hull' or 'bounding_box'
    
    Returns
    -------
    float
        Estimated area in square pixels
    """
    if len(coords) < 3:
        return np.nan
    
    if method == 'convex_hull':
        from scipy.spatial import ConvexHull
        try:
            hull = ConvexHull(coords)
            return hull.volume  # In 2D, volume = area
        except:
            # Fall back to bounding box
            method = 'bounding_box'
    
    if method == 'bounding_box':
        x_range = coords[:, 0].max() - coords[:, 0].min()
        y_range = coords[:, 1].max() - coords[:, 1].min()
        return x_range * y_range
    
    return np.nan


# ════════════════════════════════════════════════════════════════════════════
# MONTE CARLO SIMULATION FOR NULL DISTRIBUTION
# ════════════════════════════════════════════════════════════════════════════

def simulate_csr_nnd(n_points: int, area: float, n_simulations: int = 999) -> np.ndarray:
    """
    Simulate mean NND under CSR using Monte Carlo.
    
    Parameters
    ----------
    n_points : int
        Number of points to simulate
    area : float
        Area (assuming square region for simplicity)
    n_simulations : int
        Number of simulations
    
    Returns
    -------
    np.ndarray
        Array of simulated mean NNDs
    """
    if n_points < 2:
        return np.array([np.nan])
    
    # Assume square region
    side = np.sqrt(area)
    
    simulated_means = []
    for _ in range(n_simulations):
        # Generate random points
        sim_coords = np.random.uniform(0, side, size=(n_points, 2))
        # Compute NND
        nnd = compute_nearest_neighbor_distances(sim_coords)
        simulated_means.append(np.mean(nnd))
    
    return np.array(simulated_means)


def compute_monte_carlo_pvalue(observed_mean_nnd: float, simulated_means: np.ndarray, 
                               alternative: str = 'less') -> float:
    """
    Compute Monte Carlo p-value.
    
    Parameters
    ----------
    observed_mean_nnd : float
        Observed mean NND
    simulated_means : np.ndarray
        Simulated mean NNDs under null
    alternative : str
        'less' (clustered), 'greater' (dispersed), or 'two-sided'
    
    Returns
    -------
    float
        p-value
    """
    n_sim = len(simulated_means)
    
    if alternative == 'less':
        # Test for clustering (observed < expected)
        p = (np.sum(simulated_means <= observed_mean_nnd) + 1) / (n_sim + 1)
    elif alternative == 'greater':
        # Test for dispersion (observed > expected)
        p = (np.sum(simulated_means >= observed_mean_nnd) + 1) / (n_sim + 1)
    else:
        # Two-sided
        p_less = (np.sum(simulated_means <= observed_mean_nnd) + 1) / (n_sim + 1)
        p_greater = (np.sum(simulated_means >= observed_mean_nnd) + 1) / (n_sim + 1)
        p = 2 * min(p_less, p_greater)
    
    return p


# ════════════════════════════════════════════════════════════════════════════
# CORE-LEVEL ANALYSIS
# ════════════════════════════════════════════════════════════════════════════

def analyze_core_clustering(
    adata_core,
    target_cells_mask: np.ndarray,
    n_simulations: int = 999
) -> Dict:
    """
    Analyze spatial clustering for target cells in a single core.
    
    Parameters
    ----------
    adata_core : AnnData
        Subset for single core
    target_cells_mask : np.ndarray
        Boolean mask for target cells (e.g., KSHV+ CD34+ LECs)
    n_simulations : int
        Number of Monte Carlo simulations
    
    Returns
    -------
    dict
        Clustering statistics
    """
    obs = adata_core.obs
    
    # Get all cell coordinates
    if 'spatial' in adata_core.obsm:
        all_coords = adata_core.obsm['spatial'][:, :2]
    else:
        all_coords = obs[['x_centroid', 'y_centroid']].values
    
    # Get target cell coordinates
    target_coords = all_coords[target_cells_mask]
    n_target = len(target_coords)
    
    # Estimate core area from all cells
    core_area = estimate_core_area(all_coords)
    
    results = {
        'n_target_cells': n_target,
        'n_total_cells': len(all_coords),
        'core_area_px2': core_area,
        'core_area_um2': core_area / (PIXEL_PER_UM ** 2) if not np.isnan(core_area) else np.nan,
    }
    
    if n_target < 2:
        results.update({
            'observed_mean_nnd_px': np.nan,
            'observed_mean_nnd_um': np.nan,
            'expected_mean_nnd_px': np.nan,
            'expected_mean_nnd_um': np.nan,
            'clark_evans_r': np.nan,
            'clark_evans_z': np.nan,
            'clark_evans_p': np.nan,
            'mc_p_value': np.nan,
            'interpretation': 'Insufficient cells'
        })
        return results
    
    # Compute observed NND
    observed_nnd = compute_nearest_neighbor_distances(target_coords)
    observed_mean_nnd = np.mean(observed_nnd)
    
    # Compute expected NND under CSR
    expected_mean_nnd = compute_expected_nnd_csr(n_target, core_area)
    
    # Clark-Evans R index
    r_index = compute_clark_evans_r(observed_mean_nnd, expected_mean_nnd)
    
    # Clark-Evans Z statistic
    z_stat, z_pval = compute_clark_evans_z(observed_mean_nnd, n_target, core_area)
    
    # Monte Carlo simulation
    simulated_means = simulate_csr_nnd(n_target, core_area, n_simulations)
    mc_pval = compute_monte_carlo_pvalue(observed_mean_nnd, simulated_means, alternative='less')
    
    # Interpretation
    if r_index < 0.5:
        interpretation = 'Highly clustered'
    elif r_index < 0.8:
        interpretation = 'Moderately clustered'
    elif r_index < 1.2:
        interpretation = 'Random'
    elif r_index < 1.5:
        interpretation = 'Moderately dispersed'
    else:
        interpretation = 'Highly dispersed'
    
    results.update({
        'observed_mean_nnd_px': observed_mean_nnd,
        'observed_mean_nnd_um': observed_mean_nnd / PIXEL_PER_UM,
        'expected_mean_nnd_px': expected_mean_nnd,
        'expected_mean_nnd_um': expected_mean_nnd / PIXEL_PER_UM,
        'clark_evans_r': r_index,
        'clark_evans_z': z_stat,
        'clark_evans_p': z_pval,
        'mc_p_value': mc_pval,
        'interpretation': interpretation
    })
    
    return results


def process_single_core(
    adata, 
    core: str, 
    n_simulations: int = 999
) -> Dict:
    """
    Process a single core for KSHV+ CD34+ LEC clustering analysis.
    """
    adata_core = adata[adata.obs[CORE_COL] == core].copy()
    
    if len(adata_core) == 0:
        return None
    
    obs = adata_core.obs
    
    # Get metadata
    patient = obs[PATIENT_COL].iloc[0] if PATIENT_COL in obs.columns else 'Unknown'
    stage = obs[STAGE_COL].iloc[0] if STAGE_COL in obs.columns else 'Unknown'
    
    # Define target cells: KSHV+ CD34+ LECs
    lec_mask = obs[CELLTYPE_COL] == 'Lymphatic Endothelial Cells'
    infected_mask = obs[INFECTION_COL] == 'infected' if INFECTION_COL in obs.columns else np.zeros(len(obs), dtype=bool)
    cd34_pos_mask = obs[CD34_COL] == 'CD34+' if CD34_COL in obs.columns else np.ones(len(obs), dtype=bool)
    
    # KSHV+ CD34+ LECs
    target_mask = lec_mask & infected_mask & cd34_pos_mask
    
    # Also compute for comparison groups
    # KSHV+ CD34- LECs
    target_mask_cd34neg = lec_mask & infected_mask & (obs[CD34_COL] == 'CD34-') if CD34_COL in obs.columns else np.zeros(len(obs), dtype=bool)
    
    # All KSHV+ LECs
    all_infected_lec_mask = lec_mask & infected_mask
    
    # Analyze main target
    main_results = analyze_core_clustering(adata_core, target_mask.values, n_simulations)
    
    # Add metadata
    main_results['core'] = core
    main_results['patient'] = patient
    main_results['stage'] = stage
    main_results['target_type'] = 'KSHV+ CD34+ LEC'
    
    return main_results


# ════════════════════════════════════════════════════════════════════════════
# MAIN ANALYSIS FUNCTION
# ════════════════════════════════════════════════════════════════════════════

def analyze_spatial_clustering(
    adata,
    n_simulations: int = 999,
    min_cells: int = 5,
    n_jobs: int = -1,
    plot: bool = True,
    save: bool = True
) -> Dict:
    """
    Analyze spatial clustering of KSHV+ CD34+ LECs across all cores.
    
    Parameters
    ----------
    adata : AnnData
    n_simulations : int
        Number of Monte Carlo simulations per core
    min_cells : int
        Minimum target cells to include a core
    n_jobs : int
        Number of parallel jobs
    plot : bool
        Whether to generate plots
    save : bool
        Whether to save outputs
    
    Returns
    -------
    dict
        Analysis results
    """
    print("\n" + "═" * 70)
    print("SPATIAL CLUSTERING ANALYSIS: KSHV⁺ CD34⁺ LECs")
    print("═" * 70)
    
    ensure_output_dir()
    
    # Check for CD34_status column
    if CD34_COL not in adata.obs.columns:
        print(f"\n  Creating {CD34_COL} column from CD34 gene expression...")
        import scipy.sparse as sp
        cd34_counts = adata[:, 'CD34'].X
        if sp.issparse(cd34_counts):
            cd34_counts = cd34_counts.toarray()
        adata.obs[CD34_COL] = np.where(cd34_counts.squeeze() > 0, 'CD34+', 'CD34-')
        print(f"  ✓ {CD34_COL} created")
    
    # Get cores
    cores = adata.obs[CORE_COL].unique().tolist()
    print(f"\n  Processing {len(cores)} cores...")
    print(f"  Monte Carlo simulations per core: {n_simulations}")
    
    # Process cores in parallel
    results_list = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(process_single_core)(adata, core, n_simulations)
        for core in cores
    )
    
    # Filter out None results
    results_list = [r for r in results_list if r is not None]
    
    # Convert to DataFrame
    df = pd.DataFrame(results_list)
    
    # Filter to cores with sufficient cells
    df_valid = df[df['n_target_cells'] >= min_cells].copy()
    
    print(f"\n  Total cores: {len(df)}")
    print(f"  Cores with ≥{min_cells} KSHV⁺ CD34⁺ LECs: {len(df_valid)}")
    
    # Summary statistics
    print("\n" + "-" * 70)
    print("Clark-Evans R Index Summary (R < 1 = clustered)")
    print("-" * 70)
    
    for stage in DISEASE_STAGES:
        stage_df = df_valid[df_valid['stage'] == stage]
        if len(stage_df) > 0:
            r_vals = stage_df['clark_evans_r'].dropna()
            n_clustered = (r_vals < 1).sum()
            n_total = len(r_vals)
            print(f"  {stage:10s}: R = {r_vals.mean():.3f} ± {r_vals.std():.3f} "
                  f"| {n_clustered}/{n_total} cores clustered ({100*n_clustered/n_total:.1f}%)")
    
    # Overall
    r_all = df_valid['clark_evans_r'].dropna()
    n_clustered_all = (r_all < 1).sum()
    print(f"\n  Overall: R = {r_all.mean():.3f} ± {r_all.std():.3f} "
          f"| {n_clustered_all}/{len(r_all)} cores clustered ({100*n_clustered_all/len(r_all):.1f}%)")
    
    # Interpretation distribution
    print("\n" + "-" * 70)
    print("Interpretation Distribution:")
    print("-" * 70)
    print(df_valid['interpretation'].value_counts().to_string())
    
    # Package results
    results = {
        'all_cores': df,
        'valid_cores': df_valid,
        'summary_by_stage': df_valid.groupby('stage')['clark_evans_r'].agg(['mean', 'std', 'count']),
        'parameters': {
            'n_simulations': n_simulations,
            'min_cells': min_cells
        }
    }
    
    if save:
        # Save all results
        df.to_csv(f"{OUTPUT_DIR}/spatial_clustering_all_cores.csv", index=False)
        print(f"\n✓ Saved: {OUTPUT_DIR}/spatial_clustering_all_cores.csv")
        
        # Save valid cores
        df_valid.to_csv(f"{OUTPUT_DIR}/spatial_clustering_valid_cores.csv", index=False)
        print(f"✓ Saved: {OUTPUT_DIR}/spatial_clustering_valid_cores.csv")
        
        # Save summary
        summary_df = df_valid.groupby('stage').agg({
            'clark_evans_r': ['mean', 'std', 'count'],
            'observed_mean_nnd_um': ['mean', 'std'],
            'n_target_cells': ['mean', 'sum']
        }).round(3)
        summary_df.to_csv(f"{OUTPUT_DIR}/spatial_clustering_summary.csv")
        print(f"✓ Saved: {OUTPUT_DIR}/spatial_clustering_summary.csv")
    
    if plot:
        _generate_plots(results, save)
    
    print("\n" + "═" * 70)
    print("✓ SPATIAL CLUSTERING ANALYSIS COMPLETE")
    print("═" * 70)
    
    return results


# ════════════════════════════════════════════════════════════════════════════
# PLOTTING FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def _generate_plots(results: Dict, save: bool):
    """Generate all plots for spatial clustering analysis."""
    
    df = results['valid_cores']
    
    # ─────────────────────────────────────────────────────────────────────────
    # Figure 1: Clark-Evans R by stage (box plot)
    # ─────────────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    
    df_plot = df[df['stage'].isin(DISEASE_STAGES)].copy()
    df_plot['stage'] = pd.Categorical(df_plot['stage'], categories=DISEASE_STAGES, ordered=True)
    
    sns.boxplot(
        data=df_plot,
        x='stage',
        y='clark_evans_r',
        order=DISEASE_STAGES,
        palette=[STAGE_COLORS[s] for s in DISEASE_STAGES],
        ax=ax
    )
    
    # Add individual points
    sns.stripplot(
        data=df_plot,
        x='stage',
        y='clark_evans_r',
        order=DISEASE_STAGES,
        color='black',
        alpha=0.5,
        size=4,
        ax=ax
    )
    
    # Reference line at R=1 (CSR)
    ax.axhline(1, color='red', linestyle='--', linewidth=2, label='R=1 (Random)')
    ax.axhline(0.5, color='blue', linestyle=':', linewidth=1.5, label='R=0.5 (Moderately clustered)')
    
    ax.set_xlabel('Disease Stage')
    ax.set_ylabel('Clark-Evans R Index')
    ax.set_title('Spatial Clustering of KSHV⁺ CD34⁺ LECs\n(R < 1 = Clustered, supports expansion)')
    ax.legend(loc='upper right')
    
    # Add text annotation
    n_clustered = (df_plot['clark_evans_r'] < 1).sum()
    n_total = len(df_plot)
    ax.text(0.02, 0.98, f'{n_clustered}/{n_total} cores clustered ({100*n_clustered/n_total:.1f}%)',
            transform=ax.transAxes, ha='left', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    
    if save:
        path = f"{OUTPUT_DIR}/clark_evans_r_by_stage.pdf"
        plt.savefig(path, bbox_inches='tight')
        print(f"✓ Saved: {path}")
    plt.show()
    
    # ─────────────────────────────────────────────────────────────────────────
    # Figure 2: Observed vs Expected NND
    # ─────────────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 8))
    
    for stage in DISEASE_STAGES:
        stage_df = df_plot[df_plot['stage'] == stage]
        ax.scatter(
            stage_df['expected_mean_nnd_um'],
            stage_df['observed_mean_nnd_um'],
            c=STAGE_COLORS[stage],
            label=stage.capitalize(),
            alpha=0.7,
            s=50,
            edgecolor='black',
            linewidth=0.5
        )
    
    # Add diagonal line (observed = expected)
    max_val = max(df_plot['expected_mean_nnd_um'].max(), df_plot['observed_mean_nnd_um'].max())
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Observed = Expected')
    
    ax.set_xlabel('Expected Mean NND (µm) under CSR')
    ax.set_ylabel('Observed Mean NND (µm)')
    ax.set_title('Observed vs Expected Nearest-Neighbor Distance\n(Points below line = clustered)')
    ax.legend()
    ax.set_aspect('equal')
    
    plt.tight_layout()
    
    if save:
        path = f"{OUTPUT_DIR}/observed_vs_expected_nnd.pdf"
        plt.savefig(path, bbox_inches='tight')
        print(f"✓ Saved: {path}")
    plt.show()
    
    # ─────────────────────────────────────────────────────────────────────────
    # Figure 3: Distribution of R values
    # ─────────────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    
    for i, stage in enumerate(DISEASE_STAGES):
        stage_df = df_plot[df_plot['stage'] == stage]
        r_vals = stage_df['clark_evans_r'].dropna()
        
        if len(r_vals) > 0:
            ax.hist(r_vals, bins=20, alpha=0.5, color=STAGE_COLORS[stage], 
                   label=f'{stage.capitalize()} (n={len(r_vals)})', edgecolor='black')
    
    ax.axvline(1, color='red', linestyle='--', linewidth=2, label='R=1 (Random)')
    ax.set_xlabel('Clark-Evans R Index')
    ax.set_ylabel('Number of Cores')
    ax.set_title('Distribution of Spatial Clustering Index\n(R < 1 = Clustered)')
    ax.legend()
    
    plt.tight_layout()
    
    if save:
        path = f"{OUTPUT_DIR}/clark_evans_r_distribution.pdf"
        plt.savefig(path, bbox_inches='tight')
        print(f"✓ Saved: {path}")
    plt.show()
    
    # ─────────────────────────────────────────────────────────────────────────
    # Figure 4: R vs number of cells (to check for bias)
    # ─────────────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    
    for stage in DISEASE_STAGES:
        stage_df = df_plot[df_plot['stage'] == stage]
        ax.scatter(
            stage_df['n_target_cells'],
            stage_df['clark_evans_r'],
            c=STAGE_COLORS[stage],
            label=stage.capitalize(),
            alpha=0.7,
            s=50,
            edgecolor='black',
            linewidth=0.5
        )
    
    ax.axhline(1, color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Number of KSHV⁺ CD34⁺ LECs per Core')
    ax.set_ylabel('Clark-Evans R Index')
    ax.set_title('Clustering vs Cell Count\n(Check for cell-number bias)')
    ax.legend()
    ax.set_xscale('log')
    
    plt.tight_layout()
    
    if save:
        path = f"{OUTPUT_DIR}/clark_evans_r_vs_ncells.pdf"
        plt.savefig(path, bbox_inches='tight')
        print(f"✓ Saved: {path}")
    plt.show()
    
    # ─────────────────────────────────────────────────────────────────────────
    # Figure 5: Summary bar plot
    # ─────────────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left: Mean R by stage
    ax = axes[0]
    summary = df_plot.groupby('stage')['clark_evans_r'].agg(['mean', 'sem']).reindex(DISEASE_STAGES)
    
    bars = ax.bar(
        range(len(DISEASE_STAGES)),
        summary['mean'],
        yerr=summary['sem'],
        color=[STAGE_COLORS[s] for s in DISEASE_STAGES],
        edgecolor='black',
        capsize=5
    )
    
    ax.axhline(1, color='red', linestyle='--', linewidth=2, label='R=1 (Random)')
    ax.set_xticks(range(len(DISEASE_STAGES)))
    ax.set_xticklabels([s.capitalize() for s in DISEASE_STAGES])
    ax.set_ylabel('Clark-Evans R Index (mean ± SEM)')
    ax.set_title('Mean Clustering Index by Stage')
    ax.legend()
    
    # Add value labels
    for i, (bar, mean) in enumerate(zip(bars, summary['mean'])):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{mean:.2f}', ha='center', va='bottom', fontsize=10)
    
    # Right: Percentage clustered by stage
    ax = axes[1]
    pct_clustered = df_plot.groupby('stage').apply(
        lambda x: 100 * (x['clark_evans_r'] < 1).sum() / len(x)
    ).reindex(DISEASE_STAGES)
    
    bars = ax.bar(
        range(len(DISEASE_STAGES)),
        pct_clustered,
        color=[STAGE_COLORS[s] for s in DISEASE_STAGES],
        edgecolor='black'
    )
    
    ax.set_xticks(range(len(DISEASE_STAGES)))
    ax.set_xticklabels([s.capitalize() for s in DISEASE_STAGES])
    ax.set_ylabel('% Cores with Clustered Pattern')
    ax.set_title('Percentage of Cores Showing Clustering')
    ax.set_ylim(0, 105)
    
    # Add value labels
    for bar, pct in zip(bars, pct_clustered):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
               f'{pct:.0f}%', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    if save:
        path = f"{OUTPUT_DIR}/clustering_summary_barplots.pdf"
        plt.savefig(path, bbox_inches='tight')
        print(f"✓ Saved: {path}")
    plt.show()


# ════════════════════════════════════════════════════════════════════════════
# GENERATE MANUSCRIPT TEXT
# ════════════════════════════════════════════════════════════════════════════

def generate_manuscript_text(results: Dict) -> str:
    """
    Generate text snippet for manuscript based on analysis results.
    """
    df = results['valid_cores']
    
    # Overall statistics
    r_all = df['clark_evans_r'].dropna()
    n_clustered = (r_all < 1).sum()
    n_total = len(r_all)
    pct_clustered = 100 * n_clustered / n_total
    
    mean_r = r_all.mean()
    std_r = r_all.std()
    
    # Per-stage
    stage_stats = {}
    for stage in DISEASE_STAGES:
        stage_df = df[df['stage'] == stage]
        r_vals = stage_df['clark_evans_r'].dropna()
        if len(r_vals) > 0:
            stage_stats[stage] = {
                'mean_r': r_vals.mean(),
                'pct_clustered': 100 * (r_vals < 1).sum() / len(r_vals),
                'n': len(r_vals)
            }
    
    text = f"""
**Spatial clustering analysis:**

To address whether KSHV⁺ CD34⁺ LECs represent expanded populations 
or independent infection events, we performed nearest-neighbor distance (NND) 
analysis using the Clark-Evans aggregation index (R). Under this framework, 
R < 1 indicates spatial clustering (consistent with expansion), 
R ≈ 1 indicates random distribution (consistent with independent infections), 
and R > 1 indicates dispersed/regular patterns.

- Patch: R = {stage_stats.get('patch', {}).get('mean_r', 'N/A'):.2f}, {stage_stats.get('patch', {}).get('pct_clustered', 'N/A'):.0f}% clustered
- Plaque: R = {stage_stats.get('plaque', {}).get('mean_r', 'N/A'):.2f}, {stage_stats.get('plaque', {}).get('pct_clustered', 'N/A'):.0f}% clustered  
- Nodular: R = {stage_stats.get('nodular', {}).get('mean_r', 'N/A'):.2f}, {stage_stats.get('nodular', {}).get('pct_clustered', 'N/A'):.0f}% clustered

"""
    
    print(text)
    return text


# ════════════════════════════════════════════════════════════════════════════
# PRINT USAGE
# ════════════════════════════════════════════════════════════════════════════

print("""
╔══════════════════════════════════════════════════════════════════════╗
║  SPATIAL CLUSTERING ANALYSIS: KSHV⁺ CD34⁺ LECs                       ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Main function:                                                      ║
║    results = analyze_spatial_clustering(adata)                       ║
║                                                                      ║
║  Options:                                                            ║
║    n_simulations=999   Monte Carlo simulations per core              ║
║    min_cells=5         Minimum KSHV+ CD34+ LECs to analyze core      ║
║    n_jobs=-1           Use all CPUs                                  ║
║                                                                      ║
║  Generate manuscript text:                                           ║
║    text = generate_manuscript_text(results)                          ║
║                                                                      ║
║  Key output:                                                         ║
║    Clark-Evans R < 1 → Clustered (supports expansion)                ║
║    Clark-Evans R ≈ 1 → Random (supports independent infection)       ║
║    Clark-Evans R > 1 → Dispersed                                     ║
║                                                                      ║
║  Outputs saved to: figures/spatial_clustering/                       ║
╚══════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
results = analyze_spatial_clustering(adata)

# Generate manuscript text
text = generate_manuscript_text(results)

In [ ]:
adata.obs['KSHV_CD34_LECs'] = (
    (adata.obs['infection_status'] == 'infected') &
    (adata.obs['cell_type'] == 'Lymphatic Endothelial Cells') &
    (adata.obs['CD34_status'] == 'CD34+')
)

print(f"KSHV+ CD34+ LECs: {adata.obs['KSHV_CD34_LECs'].sum():,}")

In [ ]:
mask = adata.obs['KSHV_CD34_LECs']
print(f"Total KSHV+ CD34+ LECs: {mask.sum():,}")
for stage in ['patch', 'plaque', 'nodular']:
    s = (adata.obs['Stage'] == stage) & mask
    print(f"  {stage}: {s.sum():,}")

In [ ]:
# Check each filter independently
print("=== Component checks ===")
print(f"infection_status == 'infected': {(adata.obs['infection_status'] == 'infected').sum():,}")
print(f"cell_type == 'Lymphatic Endothelial Cells': {(adata.obs['cell_type'] == 'Lymphatic Endothelial Cells').sum():,}")
print(f"CD34_status == 'CD34+': {(adata.obs['CD34_status'] == 'CD34+').sum():,}")

# Check CD34_status values
print(f"\n=== CD34_status value counts ===")
print(adata.obs['CD34_status'].value_counts(dropna=False))

# Check infection_status values
print(f"\n=== infection_status value counts ===")
print(adata.obs['infection_status'].value_counts(dropna=False))

# Two-way intersections
inf = adata.obs['infection_status'] == 'infected'
lec = adata.obs['cell_type'] == 'Lymphatic Endothelial Cells'
cd34 = adata.obs['CD34_status'] == 'CD34+'

print(f"\n=== Intersections ===")
print(f"infected AND LEC: {(inf & lec).sum():,}")
print(f"infected AND CD34+: {(inf & cd34).sum():,}")
print(f"LEC AND CD34+: {(lec & cd34).sum():,}")
print(f"infected AND LEC AND CD34+: {(inf & lec & cd34).sum():,}")

In [ ]:
# Check KSHV_positive (likely the single-cell level detection)
print("=== KSHV_positive ===")
print(adata.obs['KSHV_positive'].value_counts(dropna=False))

# Check the individual KSHV components
print("\n=== KSHV latent/lytic/K2 ===")
print(f"KSHV_latent_positive True: {(adata.obs['KSHV_latent_positive'] == True).sum():,}")
print(f"KSHV_lytic_positive True: {(adata.obs['KSHV_lytic_positive'] == True).sum():,}")
print(f"KSHV_lytic_K2_positive True: {(adata.obs['KSHV_lytic_K2_positive'] == True).sum():,}")

# Check CD34 expression counts
print("\n=== CD34_counts distribution ===")
print(adata.obs['CD34_counts'].describe())
print(f"\nCD34_counts > 0: {(adata.obs['CD34_counts'] > 0).sum():,}")
print(f"CD34_counts > 1: {(adata.obs['CD34_counts'] > 1).sum():,}")
print(f"CD34_counts > 2: {(adata.obs['CD34_counts'] > 2).sum():,}")

# What the correct mask should probably be
kshv = adata.obs['KSHV_positive'] == True
lec = adata.obs['cell_type'] == 'Lymphatic Endothelial Cells'
cd34 = adata.obs['CD34_counts'] > 0  # or CD34_status, depending on above

print(f"\n=== Corrected intersection ===")
print(f"KSHV_positive AND LEC: {(kshv & lec).sum():,}")
print(f"KSHV_positive AND LEC AND CD34>0: {(kshv & lec & cd34).sum():,}")
print(f"KSHV_positive AND LEC AND CD34_status=='CD34+': {(kshv & lec & (adata.obs['CD34_status']=='CD34+')).sum():,}")

In [ ]:
# This column sounds like exactly what you need
print("=== infection_status_kshv_cd34 ===")
print(adata.obs['infection_status_kshv_cd34'].value_counts(dropna=False))

# And this one
print("\n=== broad_cell_types_with_CD34 ===")
print(adata.obs['broad_cell_types_with_CD34'].value_counts(dropna=False))

# Also check the express_CD34_only column
print("\n=== express_CD34_only ===")
print(adata.obs['express_CD34_only'].value_counts(dropna=False))

# Check what the original Figure 5 used
print("\n=== CD34_status by broad_cell_types (LECs only) ===")
lec = adata.obs['cell_type'] == 'Lymphatic Endothelial Cells'
print(adata.obs.loc[lec, 'CD34_status'].value_counts(dropna=False))

In [ ]:
lec = adata.obs['cell_type'] == 'Lymphatic Endothelial Cells'
kshv = adata.obs['KSHV_positive'] == True
cd34_only = adata.obs['express_CD34_only'] == True

print("=== express_CD34_only among LECs ===")
print(f"LECs with express_CD34_only: {(lec & cd34_only).sum():,}")
print(f"KSHV+ LECs with express_CD34_only: {(lec & kshv & cd34_only).sum():,}")

for stage in ['patch', 'plaque', 'nodular']:
    s = adata.obs['Stage'] == stage
    print(f"  {stage}: {(s & lec & kshv & cd34_only).sum():,}")

# Also check what percentage of LECs this represents
print(f"\n% of LECs: {(lec & cd34_only).sum() / lec.sum() * 100:.1f}%")

# For comparison, check the other marker combos
print("\n=== Other marker columns among LECs ===")
for col in ['express_CD34_only', 'express_ENG_only', 'express_THY1_only', 
            'express_KIT_only', 'express_CD34_THY1_KIT', 'express_ENG_THY1_KIT']:
    print(f"  {col}: {(lec & (adata.obs[col] == True)).sum():,}")

In [ ]:
import numpy as np
from scipy import sparse

# Check raw KSHV gene expression
kshv_genes = ['KSHV.ORF71', 'KSHV.ORF72', 'KSHV.ORF73',  # latent
              'KSHV.ORF50', 'KSHV.ORF57', 'KSHV.ORF59', 'KSHV.K9', 'KSHV.ORF65',  # lytic
              'KSHV.K2']  # vIL6

available = [g for g in kshv_genes if g in adata.var_names]
print(f"Available KSHV genes: {available}")

gene_idx = [adata.var_names.get_loc(g) for g in available]
X = adata.X
if sparse.issparse(X):
    kshv_expr = np.asarray(X[:, gene_idx].sum(axis=1)).flatten()
else:
    kshv_expr = X[:, gene_idx].sum(axis=1)

kshv_direct = kshv_expr > 0
print(f"\nDirect KSHV detection (any transcript > 0): {kshv_direct.sum():,}")
print(f"vs KSHV_positive column: {(adata.obs['KSHV_positive'] == True).sum():,}")

# Now check the target population
lec = adata.obs['cell_type'] == 'Lymphatic Endothelial Cells'
cd34_only = adata.obs['express_CD34_only'] == True

print(f"\nDirect KSHV+ LECs: {(kshv_direct & lec).sum():,}")
print(f"Direct KSHV+ CD34-only LECs: {(kshv_direct & lec & cd34_only).sum():,}")

for stage in ['patch', 'plaque', 'nodular']:
    s = adata.obs['Stage'] == stage
    print(f"  {stage}: {(s & kshv_direct & lec & cd34_only).sum():,}")

In [ ]:
"""
NND Histogram — Publication-quality version for Figure S[Z]D

Key improvements over v1:
- X-axis clipped to 99th percentile so distributions are visible
- KDE overlay for smooth visual comparison
- Panel letter, clean annotation of R values
- Shaded region between observed/expected means
- Arial font, Nature-style formatting
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.spatial import KDTree
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════════
# STYLE
# ════════════════════════════════════════════════════════════════
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 9,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
})

STAGE_COLORS = {
    'patch': '#2ecc71',
    'plaque': '#f1c40f',
    'nodular': '#e74c3c',
}
CSR_COLOR = '#999999'


# ════════════════════════════════════════════════════════════════
# CORE FUNCTIONS
# ════════════════════════════════════════════════════════════════
def calculate_nnd(coords):
    if len(coords) < 2:
        return np.array([])
    tree = KDTree(coords)
    distances, _ = tree.query(coords, k=2)
    return distances[:, 1]


def simulate_csr_nnd(n_points, area, n_sims=100):
    side = np.sqrt(area)
    all_nnd = []
    for _ in range(n_sims):
        pts = np.random.uniform(0, side, size=(n_points, 2))
        all_nnd.extend(calculate_nnd(pts))
    return np.array(all_nnd)


def clark_evans_r(observed_nnd, n_points, area):
    obs_mean = np.mean(observed_nnd)
    exp_mean = 0.5 / np.sqrt(n_points / area)
    return obs_mean / exp_mean, obs_mean, exp_mean


# ════════════════════════════════════════════════════════════════
# COLLECT DATA
# ════════════════════════════════════════════════════════════════
def collect_nnd_data(adata, target_mask, stage_col='Stage',
                     core_col='path_block_core',
                     x_col='x_centroid', y_col='y_centroid',
                     pixel_size=0.2125, n_sims=100,
                     stages=['patch', 'plaque', 'nodular'],
                     min_cells=5):
    """
    Compute NND data per stage.
    
    Parameters
    ----------
    adata : AnnData
    target_mask : pd.Series (bool)
        Boolean mask identifying the target cell population
    """
    stage_data = {s: {'observed': [], 'expected': [], 'R_values': []}
                  for s in stages}

    for stage in stages:
        print(f"  Processing {stage}...")
        stage_mask = adata.obs[stage_col].str.lower() == stage.lower()
        cores = adata.obs.loc[stage_mask, core_col].unique()

        for core in cores:
            mask = (adata.obs[core_col] == core) & target_mask
            if mask.sum() < min_cells:
                continue

            coords = adata.obs.loc[mask, [x_col, y_col]].values * pixel_size
            obs_nnd = calculate_nnd(coords)
            if len(obs_nnd) == 0:
                continue

            xr = coords[:, 0].max() - coords[:, 0].min()
            yr = coords[:, 1].max() - coords[:, 1].min()
            area = xr * yr
            if area == 0:
                continue

            R, _, _ = clark_evans_r(obs_nnd, len(coords), area)
            sim_nnd = simulate_csr_nnd(len(coords), area, n_sims=n_sims)

            stage_data[stage]['observed'].extend(obs_nnd)
            stage_data[stage]['expected'].extend(sim_nnd)
            stage_data[stage]['R_values'].append(R)

        n = len(stage_data[stage]['R_values'])
        mR = np.mean(stage_data[stage]['R_values']) if n > 0 else float('nan')
        print(f"    {n} cores, mean R = {mR:.3f}")

    return stage_data


# ════════════════════════════════════════════════════════════════
# PUBLICATION PLOT
# ════════════════════════════════════════════════════════════════
def plot_nnd_publication(stage_data, stages=['patch', 'plaque', 'nodular'],
                         output_path='Figure_SZ_D_nnd_histograms_v2.pdf',
                         xlim_percentile=99, panel_letter='D',
                         shared_ylim=True):
    """
    Publication-quality NND plot with filled KDE curves.
    
    Fixes over v1:
    - Shared y-axis across all panels for direct comparison
    - Filled KDE only (no histogram bars) for clean look
    - Removed shaded gap between means
    - Single shared x-axis label on middle panel
    """
    n_stages = len(stages)
    fig, axes = plt.subplots(1, n_stages, figsize=(3.2 * n_stages, 3.0),
                             sharey=shared_ylim)
    if n_stages == 1:
        axes = [axes]

    # Determine shared x-limit from all data
    all_obs = []
    all_exp = []
    for stage in stages:
        all_obs.extend(stage_data[stage]['observed'])
        all_exp.extend(stage_data[stage]['expected'])
    
    all_vals = np.concatenate([all_obs, all_exp])
    x_max = np.percentile(all_vals, xlim_percentile)
    x_max = np.ceil(x_max / 5) * 5

    # First pass: compute all KDEs to find global y-max
    kde_results = {}
    x_kde = np.linspace(0, x_max, 500)
    global_ymax = 0

    for stage in stages:
        obs = np.array(stage_data[stage]['observed'])
        exp = np.array(stage_data[stage]['expected'])
        obs_clip = obs[obs <= x_max]
        exp_clip = exp[exp <= x_max]

        if len(obs_clip) > 10 and len(exp_clip) > 10:
            kde_obs = gaussian_kde(obs_clip, bw_method=0.15)
            kde_exp = gaussian_kde(exp_clip, bw_method=0.15)
            y_obs = kde_obs(x_kde)
            y_exp = kde_exp(x_kde)
            kde_results[stage] = (y_obs, y_exp)
            global_ymax = max(global_ymax, y_obs.max(), y_exp.max())

    # Add a small buffer above tallest peak
    global_ymax *= 1.12

    # Second pass: plot
    for i, stage in enumerate(stages):
        ax = axes[i]
        obs = np.array(stage_data[stage]['observed'])
        exp = np.array(stage_data[stage]['expected'])
        R_vals = stage_data[stage]['R_values']
        mean_R = np.mean(R_vals) if len(R_vals) > 0 else float('nan')
        n_cores = len(R_vals)

        color = STAGE_COLORS.get(stage.lower(), '#4A90D9')

        # Filled KDE curves (no histogram bars)
        if stage in kde_results:
            y_obs, y_exp = kde_results[stage]

            # Expected (CSR) — gray, behind
            ax.fill_between(x_kde, y_exp, alpha=0.20, color=CSR_COLOR,
                            zorder=1)
            ax.plot(x_kde, y_exp, color=CSR_COLOR, lw=1.5,
                    linestyle='--', zorder=2)

            # Observed — colored, in front
            ax.fill_between(x_kde, y_obs, alpha=0.30, color=color,
                            zorder=3)
            ax.plot(x_kde, y_obs, color=color, lw=1.8, zorder=4)

        # Mean vertical lines
        obs_mean = np.mean(obs)
        exp_mean = np.mean(exp)
        ax.axvline(obs_mean, color=color, ls=':', lw=1.2, alpha=0.8,
                   zorder=5)
        ax.axvline(exp_mean, color=CSR_COLOR, ls=':', lw=1.2, alpha=0.8,
                   zorder=5)

        # Annotations
        ax.text(0.97, 0.95, f'R = {mean_R:.2f}',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=10, fontweight='bold')
        ax.text(0.97, 0.83,
                f'n = {len(obs):,} cells\n{n_cores} cores',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=7, color='#555555')

        # X-axis label only on middle panel
        mid = n_stages // 2
        if i == mid:
            ax.set_xlabel('Nearest neighbor distance (μm)', fontsize=9)
        else:
            ax.set_xlabel('')

        # Y-axis label only on first panel
        if i == 0:
            ax.set_ylabel('Density', fontsize=9)

        # Title
        ax.set_title(stage.capitalize(), fontsize=10, pad=6)

        # Axis formatting
        ax.set_xlim(0, x_max)
        if shared_ylim:
            ax.set_ylim(0, global_ymax)
        else:
            ax.set_ylim(bottom=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(labelsize=8)

    # Legend on last panel
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        Patch(facecolor=STAGE_COLORS['nodular'], alpha=0.35, label='Observed'),
        Line2D([0], [0], color=CSR_COLOR, lw=1.5, ls='--', label='Expected (CSR)'),
        Line2D([0], [0], color='gray', ls=':', lw=1.2, label='Mean'),
    ]
    axes[-1].legend(handles=legend_elements, loc='upper right',
                    fontsize=7, frameon=False,
                    bbox_to_anchor=(0.97, 0.72))

    # Panel letter
    if panel_letter:
        fig.text(0.01, 0.98, panel_letter, fontsize=14, fontweight='bold',
                 va='top', ha='left')

    plt.tight_layout(rect=[0.02, 0, 1, 1])
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.savefig(output_path.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {output_path}")
    print(f"Saved: {output_path.replace('.pdf', '.png')}")


# ════════════════════════════════════════════════════════════════
# RUN
# ════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    """
    Usage in Jupyter:
    
    
    """

    # 1. Define target mask (adjust as needed for your population)
    target_mask = (
        (adata.obs['KSHV_positive'] == True) &
        (adata.obs['cell_type'] == 'Lymphatic Endothelial Cells') &
        (adata.obs['CD34_status'] == 'CD34+')
    )
    print(f"Target cells: {target_mask.sum():,}")
    
    # 2. Collect NND data
    stage_data = collect_nnd_data(
        adata, target_mask,
        stage_col='Stage',
        core_col='path_block_core',
        pixel_size=0.2125,
        n_sims=100,
    )
    
    # 3. Plot
    plot_nnd_publication(
        stage_data,
        output_path='Figure_SZ_D_nnd_histograms_v2.pdf',
        xlim_percentile=99,   # clips long tail
        panel_letter='D',
    )
    pass